# SIH26027 RailPlan — Complete ML Training & Export

This notebook trains the Priority, Risk and Maintenance Duration models used by the separate ML inference API. The Node.js backend never imports Python models directly; it calls the ML service over HTTP.

The duration model uses domain-informed feature engineering. For a real railway deployment, historical-duration features must be calculated from past records only to prevent target leakage.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib seaborn joblib ortools

In [1]:
import os, joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, f1_score, classification_report

SEED=42
np.random.seed(SEED)
print("Ready")

Ready


In [2]:
# Generate domain-constrained synthetic data
N=30000
deps=["Engineering","TRD","S&T"]
assets=["Rail","Turnout","Track","OHE","Mast","Signal","Point Machine","Cable","BPAC"]
works=["Rail Joint Renewal","Track Geometry Correction","Rail Fracture Inspection","OHE Insulator Replacement","OHE Maintenance","Mast Foundation Inspection","Signal Cable Inspection","Signal Maintenance","Point Machine Maintenance","BPAC Maintenance"]
routes=["PUNE-LNL","LNL-KARJAT","KYN-PNVL","PUNE-MMR","DD-PUNE","PUNE-SOL"]
seasons=["Summer","Monsoon","Winter"]
df=pd.DataFrame({
"work_id":[f"WR-{100000+i}" for i in range(N)],"department":np.random.choice(deps,N,p=[.42,.30,.28]),"asset_type":np.random.choice(assets,N),"work_type":np.random.choice(works,N),"route":np.random.choice(routes,N),"season":np.random.choice(seasons,N,p=[.34,.34,.32]),"criticality":np.random.randint(1,5,N),"days_overdue":np.random.poisson(8,N),"failure_probability":np.random.beta(2.2,5,N),"asset_availability_impact":np.random.beta(3,2.5,N),"train_density":np.random.randint(20,105,N),"passenger_trains":np.random.randint(10,65,N),"goods_trains":np.random.randint(2,25,N),"historical_failures":np.random.poisson(2.5,N),"last_maintenance_days":np.random.randint(15,900,N),"route_importance":np.random.uniform(.35,1,N),"dependency_count":np.random.randint(0,7,N),"crew_size":np.random.randint(2,13,N),"complexity":np.random.uniform(.1,1,N),"weather_risk":np.random.uniform(0,1,N),"equipment_required":np.random.randint(1,7,N),"location_km":np.random.uniform(0,250,N),"corridor_capacity_hours":np.random.uniform(2,8,N),"train_conflict_count":np.random.randint(0,20,N),"preferred_window_hour":np.random.choice([0,1,2,3,4,5,22,23],N),"block_feasibility":np.random.uniform(.35,1,N)
})
print(df.shape)
display(df.head())

(30000, 26)


,work_id,department,asset_type,work_type,route,season,criticality,days_overdue,failure_probability,asset_availability_impact,...,dependency_count,crew_size,complexity,weather_risk,equipment_required,location_km,corridor_capacity_hours,train_conflict_count,preferred_window_hour,block_feasibility
0,WR-100000,Engineering,Signal,Rail Fracture Inspection,LNL-KARJAT,Summer,1,9,0.179915,0.798240,...,3,3,0.583119,0.237621,5,226.358608,7.720346,4,2,0.373600
1,WR-100001,S&T,Rail,OHE Maintenance,PUNE-MMR,Summer,4,8,0.128129,0.586943,...,3,5,0.906128,0.116088,6,184.850694,2.787643,2,2,0.979570
2,WR-100002,S&T,Signal,BPAC Maintenance,PUNE-LNL,Summer,2,7,0.522081,0.657734,...,5,2,0.843057,0.828668,3,249.988806,6.675388,11,1,0.685402
3,WR-100003,TRD,OHE,Rail Joint Renewal,PUNE-LNL,Monsoon,3,10,0.581720,0.619281,...,4,3,0.448451,0.019141,5,28.768888,5.826322,8,22,0.501051
4,WR-100004,Engineering,OHE,Mast Foundation Inspection,PUNE-SOL,Winter,2,8,0.435589,0.119167,...,2,5,0.974743,0.279827,1,121.587703,4.944687,16,1,0.483340


In [3]:
# Create targets
priority=(.28*df.criticality/4+.18*df.failure_probability+.18*df.asset_availability_impact+.12*df.route_importance+.10*np.minimum(df.days_overdue/30,1)+.08*np.minimum(df.train_density/100,1)+.06*df.complexity+np.random.normal(0,.02,N)).clip(0,1)
risk=(.42*df.failure_probability+.22*df.asset_availability_impact+.12*df.criticality/4+.10*df.route_importance+.08*df.complexity+.06*np.minimum(df.historical_failures/8,1)+np.random.normal(0,.03,N)).clip(0,1)
df["priority_score_target"]=priority
df["priority_class_target"]=pd.cut(priority,[-.01,.35,.60,.80,1.01],labels=["Low","Medium","High","Critical"]).astype(str)
df["risk_score_target"]=risk
df["risk_class_target"]=pd.cut(risk,[-.01,.30,.55,.78,1.01],labels=["Low","Medium","High","Critical"]).astype(str)
base={"Rail Joint Renewal":2.8,"Track Geometry Correction":3.5,"Rail Fracture Inspection":2.0,"OHE Insulator Replacement":2.5,"OHE Maintenance":3.2,"Mast Foundation Inspection":2.7,"Signal Cable Inspection":2.2,"Signal Maintenance":2.8,"Point Machine Maintenance":3.0,"BPAC Maintenance":3.4}
ad={"Rail":.3,"Turnout":.8,"Track":.7,"OHE":.6,"Mast":.5,"Signal":.4,"Point Machine":.7,"Cable":.4,"BPAC":.8}
dd={"Engineering":.3,"TRD":.4,"S&T":.35}
dur=df.work_type.map(base)+df.asset_type.map(ad)+df.department.map(dd)+1.5*df.complexity+.18*df.equipment_required+.12*df.dependency_count+.035*df.train_conflict_count+.45*df.weather_risk+.0009*df.location_km-.055*df.crew_size+.0025*df.last_maintenance_days+.10*df.historical_failures+.35*df.failure_probability+.50*df.asset_availability_impact+.15*df.route_importance+.004*df.train_density*df.complexity-.18*np.log1p(df.crew_size)+np.where(df.season=="Monsoon",.35*df.weather_risk,0)+np.random.normal(0,.18,N)
df["duration_hours_actual"]=np.clip(dur,.5,12)
df.to_csv("maintenance_training.csv",index=False)
print("Targets created")

Targets created


In [4]:
# Duration feature engineering
D=df.drop(columns=["priority_score_target","priority_class_target","risk_score_target","risk_class_target","duration_hours_actual"]).copy()
D["crew_equipment_ratio"]=D.crew_size/(D.equipment_required+1)
D["complexity_per_crew"]=D.complexity/(D.crew_size+1)
D["traffic_pressure"]=D.train_density*D.route_importance
D["conflict_pressure"]=D.train_conflict_count/(D.corridor_capacity_hours+.5)
D["maintenance_age_factor"]=np.log1p(D.last_maintenance_days)
D["failure_pressure"]=D.failure_probability*(1+D.historical_failures)
D["availability_pressure"]=D.asset_availability_impact*D.route_importance
D["dependency_pressure"]=D.dependency_count*D.complexity
D["weather_complexity"]=D.weather_risk*D.complexity
D["traffic_complexity"]=D.train_density*D.complexity
D["resource_score"]=D.crew_size*D.equipment_required
D["work_type_historical_duration"]=D.work_type.map(base).fillna(3.0)
D["asset_historical_duration"]=D.asset_type.map(ad).fillna(.5)
D["department_historical_duration"]=D.department.map(dd).fillna(.35)
print(D.shape)

(30000, 40)


In [5]:
# Train/test split and preprocessing
base_features=[c for c in df.columns if c not in ["work_id","priority_score_target","priority_class_target","risk_score_target","risk_class_target","duration_hours_actual"]]
cat=df[base_features].select_dtypes(include=["object"]).columns.tolist(); num=[c for c in base_features if c not in cat]
base_pre=ColumnTransformer([("cat",OneHotEncoder(handle_unknown="ignore",sparse_output=False),cat),("num","passthrough",num)])
X=df[base_features]; E=base_pre.fit_transform(X)
Xtr,Xte,ytr,yte=train_test_split(E,df.priority_score_target,test_size=.2,random_state=SEED)
priority_model=HistGradientBoostingRegressor(max_iter=350,learning_rate=.04,max_leaf_nodes=31,l2_regularization=1,random_state=SEED).fit(Xtr,ytr)
Xtr,Xte,ytr,yte=train_test_split(E,df.risk_class_target,test_size=.2,random_state=SEED,stratify=df.risk_class_target)
risk_model=RandomForestClassifier(n_estimators=350,max_depth=20,min_samples_leaf=2,class_weight="balanced_subsample",random_state=SEED,n_jobs=-1).fit(Xtr,ytr)
dcat=D.select_dtypes(include=["object"]).columns.tolist(); dnum=[c for c in D.columns if c not in dcat]
duration_pre=ColumnTransformer([("cat",OneHotEncoder(handle_unknown="ignore",sparse_output=False),dcat),("num","passthrough",dnum)])
DE=duration_pre.fit_transform(D)
Xt,Xv,yt,yv=train_test_split(DE,df.duration_hours_actual,test_size=.2,random_state=SEED)
duration_model=HistGradientBoostingRegressor(max_iter=500,learning_rate=.035,max_leaf_nodes=31,min_samples_leaf=15,l2_regularization=1,random_state=SEED).fit(Xt,yt)
print("All models trained")

All models trained


In [6]:
# Evaluate duration model
p=duration_model.predict(Xv)
mae=mean_absolute_error(yv,p); rmse=np.sqrt(mean_squared_error(yv,p)); r2=r2_score(yv,p)
print(f"MAE  : {mae:.4f} hours ({mae*60:.2f} min)")
print(f"RMSE : {rmse:.4f} hours")
print(f"R2   : {r2:.4f}")

MAE  : 0.1626 hours (9.76 min)
RMSE : 0.2048 hours
R2   : 0.9653


In [9]:
# 5-fold CV for duration
kf=KFold(n_splits=5,shuffle=True,random_state=SEED)
maes=[]; r2s=[]
for tr,va in kf.split(DE):
    m=HistGradientBoostingRegressor(max_iter=200,learning_rate=.035,max_leaf_nodes=31,min_samples_leaf=15,l2_regularization=1,random_state=SEED)
    m.fit(DE[tr],df.duration_hours_actual.iloc[tr]); pp=m.predict(DE[va])
    maes.append(mean_absolute_error(df.duration_hours_actual.iloc[va],pp)); r2s.append(r2_score(df.duration_hours_actual.iloc[va],pp))
print(f"CV MAE: {np.mean(maes):.4f} ± {np.std(maes):.4f} h")
print(f"CV R2 : {np.mean(r2s):.4f} ± {np.std(r2s):.4f}")

CV MAE: 0.1928 ± 0.0038 h
CV R2 : 0.9514 ± 0.0013


In [8]:
# Export artifacts used by the ML API
os.makedirs("../ml-service/models",exist_ok=True)
joblib.dump(base_pre,"../ml-service/models/base_preprocessor.joblib")
joblib.dump(priority_model,"../ml-service/models/priority_model.joblib")
joblib.dump(risk_model,"../ml-service/models/risk_model.joblib")
joblib.dump(duration_pre,"../ml-service/models/duration_preprocessor.joblib")
joblib.dump(duration_model,"../ml-service/models/duration_model.joblib")
df.to_csv("../backend/data/maintenance_training.csv",index=False)
print("Export complete")

Export complete


## Deployment architecture

`React frontend → Node.js/Express backend → ML HTTP API → scikit-learn models`

The Node backend does not import or execute Python models. The ML service is a separate process and can later be deployed independently.